# WineFox LoRA 微调（Kaggle）

基于 Qwen3.5-0.8B 的酒狐人设 LoRA 微调（基座与 LoRA 分离）。

**架构**：
- 基座模型（Qwen3.5-0.8B）与 LoRA adapter **分开保存**
- 对话时启用 LoRA（带酒狐人设），蒸馏时禁用 LoRA（基座模型）

**使用步骤**：
1. 新建 Kaggle Notebook，GPU 选择 T4 x2（免费）或 P100
2. 右侧 `Add Dataset` → 上传 `winefox_dataset.jsonl`（本仓库 [llm-finetune/winefox_dataset.jsonl](https://github.com/) 已生成）
3. 从上到下依次运行所有 Cell
4. 训练完成后，到右侧 `Output` 标签下载 `/kaggle/working/` 下的 LoRA f16 GGUF

**资源**：
- 基座模型：`unsloth/Qwen3.5-0.8B`（HF 在线拉取，仅用于训练）
- 数据集：`winefox_dataset.jsonl`（Kaggle Dataset 挂载到 `/kaggle/input/`）
- GPU：T4 即可（0.8B + 4bit + LoRA 约需 6GB 显存）

**产出**（保存到 `/kaggle/working/`）：
- `winefox-lora-f16.gguf`（LoRA adapter，f16，下载后本地量化）
- `winefox_lora_adapter/`（HF 格式 LoRA，可下载后在本地自行转换/量化）

**量化（在本机执行，需要 llama.cpp 的 `llama-quantize`，配合本地已有基座模型）**：
- `llama-quantize winefox-lora-f16.gguf winefox-lora-Q5_K_M.gguf Q5_K_M`
- `llama-quantize winefox-lora-f16.gguf winefox-lora-Q4_K_M.gguf Q4_K_M`


In [ ]:
# Cell 2: 安装依赖（Kaggle 上约 3-5 分钟）
# Kaggle 镜像已预装 torch / transformers / datasets / trl / peft / accelerate，
# 这里只需补装 unsloth（会自动带齐兼容的 trl/peft 版本）。
!pip install --upgrade pip -q
!pip install unsloth bitsandbytes gguf

# 若需要最新版 unsloth，可改用下面这行（慢但最新）：
# !pip install -q "unsloth[cu128] @ git+https://github.com/unslothai/unsloth.git"

# 验证环境
import torch
print(f'torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 3: 定位 Kaggle 输入数据集 + 输出目录
# 通过右侧「Add Dataset」上传后，文件会被挂载到 /kaggle/input/<数据集名>/ 下，这里自动查找。
import glob
import os

INPUT_GLOBS = glob.glob('/kaggle/input/**/winefox_dataset.jsonl', recursive=True)
if not INPUT_GLOBS:
    raise FileNotFoundError(
        '未找到 winefox_dataset.jsonl！\n'
        '请先点击右侧 Add Dataset → 上传 winefox_dataset.jsonl，再重新运行本 Cell。'
    )
DATASET_PATH = INPUT_GLOBS[0]
print(f'数据集路径: {DATASET_PATH}')

# 产物输出目录（Kaggle 上 /kaggle/working 会在 Output 面板中提供下载）
OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'输出目录: {OUTPUT_DIR}')

In [ ]:
# Cell 4: 加载 Qwen3.5-0.8B + LoRA 配置
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    model_name='unsloth/Qwen3.5-0.8B',
    load_in_4bit=True,
    use_gradient_checkpointing='unsloth',
)

# LoRA: 仅训 language + attention + mlp，不训 vision
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=32,
    lora_alpha=64,
    lora_dropout=0.0,
    bias='none',
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)
print('LoRA 配置完成')

In [ ]:
# Cell 5: 加载数据集
from datasets import Dataset

# 从 JSONL 加载 WineFox 数据集（每行一条 {"messages": [...]}）
train_dataset = Dataset.from_json(DATASET_PATH)
print(f'样本数: {len(train_dataset)}')
print(train_dataset[:3])

In [ ]:
# Cell 6: 训练配置 + 启动
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_dataset,
    args=SFTConfig(
        per_device_train_batch_size=16,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        warmup_ratio=0.05,
        learning_rate=2e-4,
        logging_steps=1,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=3407,
        output_dir=os.path.join(OUTPUT_DIR, 'outputs'),
        report_to='none',
        remove_unused_columns=True,
        max_length=2048,
        assistant_only_loss=True,
    ),
)

import time
t0 = time.time()
trainer.train()
print(f'\n训练完成，耗时: {(time.time()-t0)/60:.1f} 分钟')

In [ ]:
# Cell 7: 保存 LoRA adapter（不合并回基座）
# 基座模型与 LoRA 分离：对话时启用 LoRA，蒸馏时禁用 LoRA
lora_output_dir = os.path.join(OUTPUT_DIR, 'winefox_lora_adapter')
model.save_pretrained(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)
print(f'LoRA adapter 已保存到: {lora_output_dir}')

In [ ]:
# Cell 8: 转换 LoRA adapter 为 GGUF 格式（纯 Python，无需编译）
# llama.cpp 仓库自带 convert_lora_to_gguf.py，浅克隆只为取脚本，不需要编译任何 C++ 工具。
import subprocess

LLAMA_CPP_DIR = os.path.join(OUTPUT_DIR, 'llama.cpp')
if not os.path.exists(LLAMA_CPP_DIR):
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp', LLAMA_CPP_DIR],
        check=True,
    )

# 转换 LoRA adapter → GGUF（f16）
subprocess.run([
    'python', f'{LLAMA_CPP_DIR}/convert_lora_to_gguf.py',
    '--outfile', f'{OUTPUT_DIR}/winefox-lora-f16.gguf',
    lora_output_dir,
], check=True)
print('LoRA GGUF 转换完成（f16）: winefox-lora-f16.gguf')
print('量化请在本地执行: llama-quantize winefox-lora-f16.gguf winefox-lora-Q5_K_M.gguf Q5_K_M')


In [ ]:
# Cell 9: 汇总产物
print('\n=== 全部产出（/kaggle/working，请在右侧 Output 面板下载） ===')
print(f'LoRA GGUF（f16，下载后本地量化）: {OUTPUT_DIR}/winefox-lora-f16.gguf')
print(f'LoRA adapter 原始文件（HF 格式）: {OUTPUT_DIR}/winefox_lora_adapter/')
print('\n本地量化示例（需本机已构建 llama.cpp 的 llama-quantize）：')
print('  llama-quantize winefox-lora-f16.gguf winefox-lora-Q5_K_M.gguf Q5_K_M')
print('  llama-quantize winefox-lora-f16.gguf winefox-lora-Q4_K_M.gguf Q4_K_M')
